In [1]:
import os
import pandas as pd
from pathlib import Path
from patent_retrieval import utils as utils, dataset as dataset

In [2]:
from patent_retrieval.dataset import parse_patent as parse_patent
from tqdm import tqdm

In [3]:
test_topics_path=(Path(os.environ["CLEF_IP_LOCATION"])/ "02_topics"/ "test-pac"/ "relass_clef-ip-2011-PAC_abs.txt")
candidates_path=Path("/home/alm3rng/patent-retrieval/embeddings/runs/patQwen3-emb-4b-v2_db-v4_all-topics_abstract-claims_aysm_top1000/results.csv")

In [4]:
candidates_df = pd.read_csv(candidates_path)
candidates_df.head()

,topic,number,score
0,EP-1221372-A2,WO-2002004219,0.820527
1,EP-1221372-A2,EP-1029674,0.798651
2,EP-1221372-A2,EP-0703079,0.793282
3,EP-1221372-A2,EP-1514688,0.789982
4,EP-1221372-A2,EP-1193065,0.787915


## Sample 9 topics

In [4]:
topics_9 = [" EP-1223076-A1"," EP-1328127-A1"," EP-1467369-A1"," EP-1474992-A1"," EP-1547481-A1"," EP-1474992-A1"," EP-1662788-A1"," EP-1552963-A1"," EP-1690995-A1"]

## Sample 30 topics

In [5]:
unique_topics = candidates_df['topic'].unique()
n = min(30, len(unique_topics))
topics_30 = pd.Series(unique_topics).sample(n, random_state=42).tolist()
sample_30topics_df = candidates_df[candidates_df['topic'].isin(topics_30)].reset_index(drop=True)
sample_30topics_df

,topic,number,score
0,EP-1243277-A1,EP-1192953,0.950939
1,EP-1243277-A1,EP-1201254,0.849311
2,EP-1243277-A1,EP-0880163,0.774603
3,EP-1243277-A1,EP-0840349,0.763149
4,EP-1243277-A1,EP-0432573,0.762457
...,...,...,...
29995,EP-1936750-A1,EP-0686798,0.651722
29996,EP-1936750-A1,EP-0690325,0.651721
29997,EP-1936750-A1,EP-0824280,0.651696
29998,EP-1936750-A1,EP-0106518,0.651548


In [7]:
topics_30

['EP-1452422-A1',
 'EP-1394086-A1',
 'EP-1580034-A1',
 'EP-1249251-A1',
 'EP-1403982-A1',
 'EP-1260334-A1',
 'EP-1600824-A1',
 'EP-1452567-A1',
 'EP-1936750-A1',
 'EP-1510785-A1',
 'EP-1520990-A2',
 'EP-1484206-A1',
 'EP-1536380-A1',
 'EP-1802174-A1',
 'EP-1589224-A1',
 'EP-1759605-A2',
 'EP-1243277-A1',
 'EP-1742475-A1',
 'EP-1444944-A2',
 'EP-1612985-A1',
 'EP-1282090-A1',
 'EP-1637693-A1',
 'EP-1413530-A1',
 'EP-1262927-A1',
 'EP-1364707-A1',
 'EP-1431110-A1',
 'EP-1394311-A1',
 'EP-1424667-A2',
 'EP-1621102-A2',
 'EP-1803777-A2']

In [6]:
sample_30topics_df.to_csv("/home/alm3rng/patent-retrieval/embeddings/runs/patQwen3-emb-4b-v2_db-v4_all-topics_abstract-claims_aysm_top1000/30topics_results.csv", index=False)

## Sample 500 topics

In [6]:
test_df = utils.load_topics(test_topics_path)

In [7]:
labels_df = pd.read_csv(
    test_topics_path,
    sep="\t",
    header=None,
    names=["topic", "candidate", "label"]
)

In [41]:
import random

# Read the 30 topics from file
with open("/home/alm3rng/patent-retrieval/src/patent_retrieval/dataset/topics/30topics.txt", "r") as f:
    topics_from_file = [line.strip() for line in f if line.strip()]
    # Parse topics from test_df instead of reading test_topics_path
    
    topics_rel_counts = labels_df.groupby("topic").size().to_dict()


    # Filter topics with <= 15 relevant docs
    eligible_topics = {t for t, count in topics_rel_counts.items() if count <= 20}

    # Exclude topics already in the 30 topics file
    topics_from_file_set = set(topics_from_file)
    remaining_eligible = [t for t in eligible_topics if t not in topics_from_file_set]

    # Sample 470 from the remaining eligible topics
    sample_n = min(470, len(remaining_eligible))
    #random.shuffle(remaining_eligible)
    additional_topics = pd.Series(remaining_eligible).sample(sample_n, random_state=6543).tolist()#986745

    # Combine to get 500 topics
    topics_500 = topics_from_file + additional_topics
    print(f"Total topics: {len(topics_500)}")
    print(f"Topics from file: {len(topics_from_file)}")
    print(f"Additional sampled: {len(additional_topics)}")

output_path = Path("/home/alm3rng/patent-retrieval/src/patent_retrieval/dataset/topics/500topics.txt")
with open(output_path, "w") as out_f:
    out_f.write("\n".join(topics_500) + "\n")

print(f"Saved {len(topics_500)} topics to {output_path}")


Total topics: 500
Topics from file: 30
Additional sampled: 470
Saved 500 topics to /home/alm3rng/patent-retrieval/src/patent_retrieval/dataset/topics/500topics.txt


In [42]:
import plotly.express as px
# Plot distribution of relevant documents per topic


relevant_counts = (
    labels_df[
        labels_df["topic"].isin(topics_500) & (labels_df["label"] > 0)
    ]
    .groupby("topic")
    .size()
    .reindex(topics_500, fill_value=0)
)

fig = px.histogram(
    x=relevant_counts.values,
    nbins=30,
    title='Distribution of Relevant Documents per Topic',
    labels={'x': 'Number of Relevant Documents', 'y': 'Number of Topics'},
    opacity=0.7
)
fig.update_layout(
    xaxis_title='Number of Relevant Documents',
    yaxis_title='Frequency',
    showlegend=False
)
fig.show()

print(f"Total topics: {len(relevant_counts)}")
print(f"Avg relevant docs per topic: {relevant_counts.mean():.2f}")
print(f"Min: {relevant_counts.min()}, Max: {relevant_counts.max()}")
print(f"Median: {relevant_counts.median()}")

Total topics: 500
Avg relevant docs per topic: 4.70
Min: 1, Max: 20
Median: 4.0


In [43]:
document_collection_dir=Path(os.environ["CLEF_IP_LOCATION"]) / "01_document_collection" / "document_collection_pac"

def parse_candidate_path(patent_id: str): 

    if patent_id[:2] == 'EP':
        return f"{patent_id[:2]}/{"000000" if patent_id[3] == '0' else "000001"}/{patent_id[4:6]}/{patent_id[6:8]}/{patent_id[8:10]}/{patent_id}*.xml"
    else:
        return f"{patent_id[:2]}/00{patent_id[3:7]}/{patent_id[7:9]}/{patent_id[9:11]}/{patent_id[11:13]}/{patent_id}*.xml"


def find_patent_file(identifier: str) :
    # first try the test topics PAC_topics/files (same approach as get_patent)

    topic_glob = list(test_topics_path.parent.glob(f"PAC_topics/files/{identifier}*.xml"))

    if topic_glob:
        return topic_glob[0]
    # then try the document collection dirs (document_collection_dir is a tuple)

    topic_glob = list(document_collection_dir.glob(parse_candidate_path(identifier)))
    #print(parse_candidate_path(identifier))
    #print(topic_glob)
    #print(topic_glob)
    return topic_glob[0]


In [44]:
topics_500_path = Path("/home/alm3rng/patent-retrieval/src/patent_retrieval/dataset/topics/500topics.txt")
topics_500 = [
    line.strip()
    for line in topics_500_path.read_text(encoding="utf-8").splitlines()
    if line.strip() and not line.strip().startswith("#")
]

#topics_500_df = pd.DataFrame({"topic": topics_500})
topic_languages = {}

for topic in tqdm(topics_500):
    topic_languages[topic] = parse_patent([find_patent_file(topic)])[0].language
topic_languages

100%|██████████| 500/500 [01:09<00:00,  7.21it/s]


{'EP-1452422-A1': 'FR',
 'EP-1394086-A1': 'DE',
 'EP-1580034-A1': 'DE',
 'EP-1249251-A1': 'DE',
 'EP-1403982-A1': 'EN',
 'EP-1260334-A1': 'FR',
 'EP-1600824-A1': 'EN',
 'EP-1452567-A1': 'EN',
 'EP-1936750-A1': 'DE',
 'EP-1510785-A1': 'EN',
 'EP-1520990-A2': 'EN',
 'EP-1484206-A1': 'FR',
 'EP-1536380-A1': 'EN',
 'EP-1802174-A1': 'DE',
 'EP-1589224-A1': 'DE',
 'EP-1759605-A2': 'DE',
 'EP-1243277-A1': 'EN',
 'EP-1742475-A1': 'FR',
 'EP-1444944-A2': 'DE',
 'EP-1612985-A1': 'FR',
 'EP-1282090-A1': 'FR',
 'EP-1637693-A1': 'FR',
 'EP-1413530-A1': 'FR',
 'EP-1262927-A1': 'FR',
 'EP-1364707-A1': 'FR',
 'EP-1431110-A1': 'FR',
 'EP-1394311-A1': 'DE',
 'EP-1424667-A2': 'EN',
 'EP-1621102-A2': 'FR',
 'EP-1803777-A2': 'DE',
 'EP-1595940-A1': 'EN',
 'EP-1746183-A1': 'DE',
 'EP-1508384-A2': 'EN',
 'EP-1935503-A1': 'FR',
 'EP-1574231-A1': 'EN',
 'EP-1284321-A1': 'FR',
 'EP-1619441-A1': 'FR',
 'EP-1580185-A1': 'FR',
 'EP-1840940-A1': 'DE',
 'EP-1293751-A1': 'FR',
 'EP-1324317-A1': 'EN',
 'EP-1377050-A1'

In [45]:
topic_language_counts = {}
for lang in ['DE', 'EN', 'FR']:
    topic_language_counts[lang] = sum(1 for tl in topic_languages.values() if tl == lang)

percentages = [f"{(count / len(topic_languages)) * 100:.1f}%" for count in topic_language_counts.values()]

fig = px.bar(
    x=list(topic_language_counts.keys()),
    y=list(topic_language_counts.values()),
    text=percentages,
    labels={'x': 'Language', 'y': 'Number of Topics'},
    width=500,
    height=400,
    color_discrete_sequence=['teal']
)
fig.update_traces(textposition='outside', width=0.4)
fig.update_layout(
    xaxis_title='Language',
    yaxis_title='Number of Topics',
    showlegend=False,
    yaxis_range=[0, max(topic_language_counts.values()) * 1.15]
)
fig.show()